# Web Server

In [ ]:
import flask

import werkzeug

import psycopg2

import json

import uuid

import hashlib

import redis


# Lab: Stateful Web API Server With Client Calls Using Python

## HTTP / HTTPS are stateless by default

## Web Servers and Web API Servers use HTTP / HTTPS, so they are stateless by default

## Stateless allows scale up

## We need to make a stateful web server

## Recall that cookies are passed using headers;  web servers send clients a header telling them to set a cookie; web clients send their cookies back to the server in the header with each request (get, post, etc.)



## A simple landing page that checks to see if a SID (session id) cookie exists;  if the SID cookie does not exist, it creates one; subsequent calls to the landing page will send the SID;  note the web browsers hold cookies even after the browser is closed, even after the computer is turned off; that is how websites track you for days, weeks, months, or even years!

## uuid is a Python module to implement the UUID (universally unique identifier) according to the standard RFC 4122; getnode() gets the MAC (medium access control - not related to Apple - predates Apple);  uuid4 gets a unique random number; MAC + uuid4 will be unique;  every computer or device that networks has a hardware set MAC; MACs can be faked;

## sha256 is a 256 bit hash


In [ ]:
def my_create_sid():
    "create a SID based on mac address, a uuid number, concatenated, utf-8 encoded, and sha256 hashed"
    
    mac = uuid.getnode()
    
    universal_unique_id = uuid.uuid4()
    
    concatenated_string = str(mac) + str(universal_unique_id)
    
    sha256_string = hashlib.sha256(concatenated_string.encode('utf-8')).hexdigest()
    
    return sha256_string
    

In [ ]:
app = flask.Flask(__name__,
                  static_url_path="")

@app.route("/")
def landing_page():
    
    cookie = flask.request.cookies.get("SID")
    
    if cookie != None:
        response = flask.make_response("SID cookie exists and it's value is " + cookie)
        
    else:
        
        sid = my_create_sid()
        
        response = flask.make_response("SID cookie does not exist, creating the cookie, and setting it to " +  sid)
    
        response.set_cookie('SID', sid)
    
    return response



### We are running several web servers for this lab.  For reference, this is web_server.ipynb A

In [ ]:
werkzeug.serving.run_simple(hostname="0.0.0.0", 
                            port=443, 
                            application=app,
                            ssl_context=("w205_cert.pem","w205.key"),
                            use_debugger=True)

## Run the client side code from web_client Jupyter Notebook


## Create a web server implementing routes for login, get products, and logout; unless someone is logged in, they cannot get the products;

## We will use Postgres to hold the table of users and passwords; this is typical to use a transactional SQL relational database

## We will use Redis to hold the session data; this is typical use of Redis where it shines; in memory, scale out, NoSQL key / value is a perfect technology for session data; actually the original use case!


In [ ]:
connection = psycopg2.connect(
    user = "postgres",
    password = "ucb",
    host = "postgres",
    port = "5432",
    database = "postgres"
)

In [ ]:
cursor = connection.cursor()

In [ ]:
def my_query_products():
    "query the products from Postgres and return a Python list of products"
    
    connection.rollback()

    query = """
    
    select p.product_id, p.description, sum(quantity), sum(quantity * 12)
    from products p
         join line_items l
             on p.product_id = l.product_id
    group by p.product_id, p.description
    order by p.product_id
    
    """
    
    cursor.execute(query)
    
    rows = cursor.fetchall()

    connection.rollback()
    
    products_list = []
    
    for row in rows:
        
        products_list.append([row[0], row[1], f'{row[2]:,}', f'{row[3]:,}'])
        
    return(products_list)

In [ ]:
connection.rollback()

query = """

drop table if exists web_api_users;

create table web_api_users (
    username varchar(32),
    password_sha256 varchar(256)
);

"""

cursor.execute(query)

connection.commit()


In [ ]:
connection.rollback()


query = "insert into web_api_users values(%s, %s);"

cursor.execute(query, ("user_1", hashlib.sha256("password_1".encode('utf-8')).hexdigest()))
cursor.execute(query, ("user_2", hashlib.sha256("password_2".encode('utf-8')).hexdigest()))
cursor.execute(query, ("user_3", hashlib.sha256("password_3".encode('utf-8')).hexdigest()))
cursor.execute(query, ("user_4", hashlib.sha256("password_4".encode('utf-8')).hexdigest()))
cursor.execute(query, ("user_5", hashlib.sha256("password_5".encode('utf-8')).hexdigest()))

connection.commit()


In [ ]:
connection.rollback()


query = """
            
    select * 
    from web_api_users
    order by 1

"""
cursor.execute(query)

connection.rollback()

rows = cursor.fetchall()

for row in rows:
    print(row)


In [ ]:
def validate_login(username, password):
    "given a username and password, return True if login is valid, False otherwise"
    
    password_sha256 = hashlib.sha256(password.encode('utf-8')).hexdigest()
    
    connection.rollback()
    
    query = """
    
        select *
        from web_api_users 
        where username = %s and password_sha256 = %s
    
    """
    
    cursor.execute(query, (username, password_sha256))
    
    connection.rollback()
    
    return cursor.rowcount != 0

## For Redis we will use database 10 so we don't conflict with our earlier use of Redis; 


In [ ]:
session_db = redis.Redis(host='redis', port=6379, db=10)

In [ ]:
session_db.flushdb()

## Implement the web server

## / for static content that does not require a login

## /api/login to validate the username and password; create a SID; create a Redis key / value pair with SID as the key and username as the value; 

## /api/logout to delete the SID from Redis

## /api/product to validate the user is logged in by checking the SID in Redis; query and return products


In [ ]:
app = flask.Flask(__name__,
                  static_url_path="")

@app.route("/")
def landing_page():
    return flask.send_from_directory("static", "index.html")


@app.route("/api/login", methods=["POST"])
def api_login():
    
    username = flask.request.form['username']
    password = flask.request.form['password']
    
    if validate_login(username, password):
        
        sid = my_create_sid()
        
        session_db.set(sid, username)
        
        return_json = { "status": "success",
                        "sid": sid}
        
    else:
        
        return_json = { "status": "fail",
                        "description": "invalid username and/or password"}
    
    return(json.dumps(return_json))

 
@app.route("/api/logout", methods=["POST"])
def api_logout():
    
    sid = flask.request.form['sid']
    
    if session_db.get(sid) == None:
        
        return_json = { "status": "fail",
                        "description": "not logged in"}
    
    else: 
    
        session_db.delete(sid)

        return_json = { "status": "success" }
    
    return(json.dumps(return_json))


@app.route("/api/products", methods=["POST"])
def api_products():
    
    sid = flask.request.form['sid']
    
    if session_db.get(sid) == None:
        
        return_json = { "status": "fail",
                        "description": "not logged in"}
    
    else: 
        
        products_list = my_query_products()

        products_json_list = []

        for product in products_list:

            p = {}
            p["product_id"] = str(product[0])
            p["product_name"] = product[1]
            p["quantity"] = str(product[2])
            p["total_sales"] = str(product[3])

            products_json_list.append(p)
            
        return_json = { "status": "success",
                        "products": products_json_list}

    return(json.dumps(return_json))

### We are running several web servers for this lab.  For reference, this is web_server.ipynb B

In [ ]:
werkzeug.serving.run_simple(hostname="0.0.0.0", 
                            port=443, 
                            application=app,
                            ssl_context=("w205_cert.pem","w205.key"),
                            use_debugger=True)

## You try it - add a route for /api/stores to check for login and return the stores data from last week; solution provided in web_server_solutions and web_client_solutions